# MTG Repeat-Sales Price Index

**Research question:** What return would an investor have earned holding a representative basket of MTG singles?

**Methodology:** Three repeat-sales index constructions, from simple to rigorous:
- **Method A** — Chained Geometric Mean (sanity check)
- **Method B** — Bailey-Muth-Nourse (BMN) Adjacent-Pair OLS (primary index)
- **Method C** — Case-Shiller All-Pairs WLS with variance correction (most rigorous)

**Reference:** `docs/domain/MTG_RSI_METHODOLOGY.md` for full mathematical derivations.  
**Spec:** `docs/superpowers/specs/2026-05-24-mtg-repeat-sales-index-design.md`

---
**Data scope:** TCGPlayer · NM · Nonfoil · cards ≥ $1 · weekly aggregation

## 0. Setup & Imports

In [1]:
import matplotlib
matplotlib.use('Agg')
import os
import warnings
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from itertools import combinations
from scipy.sparse import lil_matrix, diags
from scipy.sparse.linalg import lsqr

warnings.filterwarnings('ignore')

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

REFRESH = False  # True = re-query DB; False = load from parquet cache

# --- Index parameters ---
MIN_PRICE_CENTS   = 100    # $1.00 floor
MIN_WEEKS         = 2      # card must appear at least this many weeks
MAX_LOG_JUMP      = np.log(10)  # exclude week-over-week changes > 10×
BASE_INDEX        = 100.0
MIN_CARDS_BASE    = 50     # minimum active cards to qualify as base period
MAX_OBS_C         = 52     # cards with more weeks than this get pair subsampling in method C

# --- DB connection ---
DB_CONFIG = dict(
    host='localhost', port=5433, dbname='automana', user='automana_admin',
    password=os.environ.get('AUTOMANA_DB_PASSWORD', ''),
)

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

def query_to_df(sql, params=None):
    with get_conn() as conn:
        cur = conn.cursor()
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)

print('Setup complete.')

Setup complete.


## 1. Data Extraction

Query `pricing.print_price_daily` for NM nonfoil TCGPlayer prices, aggregated to ISO weeks.

Each row = one card-week observation. The key join path is:

```
print_price_daily → card_version → unique_cards_ref (name)
                  → sets (set_code, release_date)
                  → rarity_ref (rarity)
```

In [2]:
CACHE_PATH = DATA_DIR / 'rsi_weekly_panel.parquet'

SQL_WEEKLY_PANEL = """
SELECT
    DATE_TRUNC('week', ppd.price_date)::DATE   AS price_week,
    cv.card_version_id::text                   AS card_version_id,
    ucr.card_name,
    s.set_code,
    s.released_at AS release_date,
    LOWER(r.rarity_name)                      AS rarity,
    AVG(ppd.list_avg_cents)::integer           AS list_avg_cents
FROM  pricing.print_price_daily ppd
JOIN  pricing.price_source       ps  ON ps.source_id      = ppd.source_id
JOIN  pricing.card_condition     cc  ON cc.condition_id   = ppd.condition_id
JOIN  card_catalog.card_finished cf  ON cf.finish_id      = ppd.finish_id
JOIN  card_catalog.card_version  cv  ON cv.card_version_id = ppd.card_version_id
JOIN  card_catalog.unique_cards_ref ucr ON ucr.unique_card_id = cv.unique_card_id
JOIN  card_catalog.sets          s   ON s.set_id          = cv.set_id
LEFT JOIN card_catalog.rarities_ref r ON r.rarity_id      = cv.rarity_id
WHERE ps.code = 'tcg'
  AND cc.code = 'NM'
  AND cf.code = 'NONFOIL'
  AND ppd.list_avg_cents IS NOT NULL
GROUP BY
    DATE_TRUNC('week', ppd.price_date),
    cv.card_version_id, ucr.card_name,
    s.set_code, s.released_at, r.rarity_name
ORDER BY price_week, card_version_id
"""

if REFRESH or not CACHE_PATH.exists():
    print('Querying database...')
    raw = query_to_df(SQL_WEEKLY_PANEL)
    raw['price_week'] = pd.to_datetime(raw['price_week'])
    raw.to_parquet(CACHE_PATH)
    print(f'Cached {len(raw):,} rows to {CACHE_PATH}')
else:
    raw = pd.read_parquet(CACHE_PATH)
    print(f'Loaded {len(raw):,} rows from cache')

print(f'\nDate range : {raw.price_week.min().date()} → {raw.price_week.max().date()}')
print(f'Unique cards: {raw.card_version_id.nunique():,}')
print(f'Unique weeks: {raw.price_week.nunique():,}')
print(f'\nPrice distribution (cents):')
print(raw.list_avg_cents.describe(percentiles=[.05,.25,.5,.75,.95]).to_string())

Loaded 2,240,807 rows from cache

Date range : 2025-11-10 → 2026-05-18


Unique cards: 84,423
Unique weeks: 28

Price distribution (cents):
count    2.240807e+06
mean     4.384671e+02
std      4.953888e+03
min      1.000000e+00
5%       5.000000e+00
25%      1.400000e+01
50%      2.900000e+01
75%      1.170000e+02
95%      1.293000e+03
max      5.142020e+05


In [3]:
# --- Apply filters ---

# 1. Price floor
df = raw[raw['list_avg_cents'] >= MIN_PRICE_CENTS].copy()
print(f'After $1 floor   : {len(df):,} rows, {df.card_version_id.nunique():,} cards')

# 2. Log-transform price
df['log_price'] = np.log(df['list_avg_cents'])

# 3. Require at least MIN_WEEKS observations per card
obs_count = df.groupby('card_version_id')['price_week'].nunique()
valid_cards = obs_count[obs_count >= MIN_WEEKS].index
df = df[df['card_version_id'].isin(valid_cards)].copy()
print(f'After min {MIN_WEEKS} weeks: {len(df):,} rows, {df.card_version_id.nunique():,} cards')

# 4. Sort for diff operations
df = df.sort_values(['card_version_id', 'price_week']).reset_index(drop=True)

# 5. Compute week-over-week log delta (for Methods A and B)
df['delta_log'] = df.groupby('card_version_id')['log_price'].diff()

# 6. Apply jump guard — scale threshold by actual gap (cards may skip weeks)
df['week_gap'] = df.groupby('card_version_id')['price_week'].diff().dt.days.div(7).fillna(1.0)
outlier_mask = df['delta_log'].abs() > MAX_LOG_JUMP * df['week_gap']
print(f'\nOutlier pairs excluded: {outlier_mask.sum():,} ({100*outlier_mask.mean():.2f}%)')
df.loc[outlier_mask, 'delta_log'] = np.nan

# 7. Assign integer week index (0 = earliest week in filtered data)
all_weeks = sorted(df['price_week'].unique())
week_to_k = {w: i for i, w in enumerate(all_weeks)}
df['week_k'] = df['price_week'].map(week_to_k)
n_weeks = len(all_weeks)

print(f'\nWeeks in index     : {n_weeks}')
print(f'First week         : {all_weeks[0].date()}')
print(f'Last week          : {all_weeks[-1].date()}')

After $1 floor   : 601,571 rows, 26,450 cards


After min 2 weeks: 600,556 rows, 25,435 cards



Outlier pairs excluded: 12 (0.00%)

Weeks in index     : 28
First week         : 2025-11-10
Last week          : 2026-05-18


## 2. Method A — Chained Geometric Mean

**Intuition:** Each week, look at all cards present in both this week and last week. Compute the geometric mean of their price ratios. Chain these means into a cumulative index.

**Formula:**

Let Ω(t) = cards with prices in both week t and t−1.

$$r(t) = \frac{1}{|\Omega(t)|} \sum_{i \in \Omega(t)} \left[ \ln P(i,t) - \ln P(i,t-1) \right]$$

$$\text{Index}_A(t) = 100 \times \exp\left( \sum_{s=1}^{t} r(s) \right)$$

**Limitation:** Equal-weights a $1 card and a $500 card. Used here as a sanity check.

In [4]:
# Method A: Chained Geometric Mean

pairs_a = df.dropna(subset=['delta_log'])

# Weekly mean log return + card count
weekly_a = (
    pairs_a
    .groupby('price_week')
    .agg(
        mean_delta_log=('delta_log', 'mean'),
        n_cards=('card_version_id', 'nunique'),
    )
    .reset_index()
    .sort_values('price_week')
)

# Chain: cumulative sum of log returns
weekly_a['cum_log'] = weekly_a['mean_delta_log'].cumsum()

# Anchor to base period (first week with enough cards)
base_row = weekly_a[weekly_a['n_cards'] >= MIN_CARDS_BASE].iloc[0]
base_week_a = base_row['price_week']
base_cum_log_a = base_row['cum_log']

weekly_a['index_A'] = BASE_INDEX * np.exp(weekly_a['cum_log'] - base_cum_log_a)

# Flag low-confidence weeks
weekly_a['low_confidence'] = weekly_a['n_cards'] < 10

print(f'Base period (Method A): {base_week_a.date()}')
print(f'Weeks computed        : {len(weekly_a):,}')
print(f'Low-confidence weeks  : {weekly_a.low_confidence.sum()}')
print(f'\nIndex range: {weekly_a.index_A.min():.1f} – {weekly_a.index_A.max():.1f}')
print(f'Latest value: {weekly_a.index_A.iloc[-1]:.1f} (base = {BASE_INDEX})')

Base period (Method A): 2025-11-17
Weeks computed        : 27
Low-confidence weeks  : 0

Index range: 100.0 – 120.0
Latest value: 117.2 (base = 100.0)


In [5]:
# Plot Method A
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(weekly_a['price_week'], weekly_a['index_A'],
         color='#546e7a', lw=1.8, label='Method A — Chained Geometric Mean')
ax1.axhline(BASE_INDEX, color='#aaa', lw=0.8, ls='--')
ax1.axvline(base_week_a, color='#aaa', lw=0.8, ls='--')
ax1.set_ylabel('Index (base = 100)', fontsize=11)
ax1.set_xlabel('')
ax1.set_title('MTG Repeat-Sales Price Index — Method A: Chained Geometric Mean', fontsize=13)
ax1.legend(loc='upper left')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))

# Coverage bar on secondary axis
ax2 = ax1.twinx()
ax2.bar(weekly_a['price_week'], weekly_a['n_cards'],
        width=5, alpha=0.15, color='#546e7a', label='Cards (right)')
ax2.set_ylabel('Cards contributing', fontsize=9, color='#546e7a')
ax2.tick_params(axis='y', labelcolor='#546e7a')

plt.tight_layout()
plt.savefig(DATA_DIR / 'rsi_method_A.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Method B — BMN Adjacent-Pair OLS

**Intuition:** Treat the entire history as a single regression. Each week-over-week price change is one observation. The unknown parameters are the index values themselves — one per week.

**The design matrix** X has one row per adjacent pair and one column per week (excluding the base week). Row for card i going from week k−1 to k:

```
X[row, k-1] = +1   (if k-1 > 0)
X[row, k-2] = -1   (if k-2 >= 0)
```

**OLS:** β̂ = lstsq(X, y) where y = vector of Δln P values.

**Index:** Index_B(t) = 100 × exp([0, β̂₁, β̂₂, ...])

**Key advantage over A:** Cards with gaps in their history still contribute constraints on the weeks they ARE observed. The regression uses all available data simultaneously and handles unbalanced panels correctly.

In [6]:
# Method B: BMN Adjacent-Pair OLS
#
# Build sparse design matrix X of shape (n_pairs, n_weeks-1).
# Columns represent log-index values β₁..β_{T-1} (β₀ = 0 by definition).
# Row for pair (k-1 → k): +1 at column k-1, -1 at column k-2  (0-indexed columns = β₁..β_{T-1})

pairs_b = df.dropna(subset=['delta_log', 'week_k']).copy()
pairs_b = pairs_b[pairs_b['week_k'] > 0]  # need a prior week
pairs_b = pairs_b.reset_index(drop=True)

n_pairs_b = len(pairs_b)
n_cols    = n_weeks - 1  # columns = β₁ .. β_{T-1}

print(f'Adjacent pairs: {n_pairs_b:,}')
print(f'Design matrix : {n_pairs_b:,} × {n_cols:,} (sparse)')

X_b = lil_matrix((n_pairs_b, n_cols), dtype=np.float64)

for row_idx, (_, row) in enumerate(pairs_b.iterrows()):
    k = int(row['week_k'])   # current week index
    # column for β_k is k-1 (since β₁ is column 0)
    if k - 1 >= 0:           # +1 for current period β_k
        X_b[row_idx, k - 1] = +1.0
    if k - 2 >= 0:           # -1 for previous period β_{k-1}
        X_b[row_idx, k - 2] = -1.0

X_b_csr = X_b.tocsr()
y_b = pairs_b['delta_log'].values

print('Solving OLS via lsqr...')
result_b = lsqr(X_b_csr, y_b, show=False)
beta_b   = result_b[0]

log_index_b = np.concatenate([[0.0], beta_b])

# Find base week: first with >= MIN_CARDS_BASE contributing pairs
n_cards_per_week_b = pairs_b.groupby('week_k')['card_version_id'].nunique()
valid_base_weeks   = n_cards_per_week_b[n_cards_per_week_b >= MIN_CARDS_BASE].index
base_k_b           = int(valid_base_weeks.min())
base_week_b        = all_weeks[base_k_b]

log_index_b_anchored = log_index_b - log_index_b[base_k_b]
index_b_values       = BASE_INDEX * np.exp(log_index_b_anchored)

index_b = pd.DataFrame({
    'price_week':  all_weeks,
    'week_k':      range(n_weeks),
    'index_B':     index_b_values,
    'n_pairs':     [n_cards_per_week_b.get(k, 0) for k in range(n_weeks)],
})

print(f'Base period (Method B): {base_week_b.date()}')
print(f'\nIndex range: {index_b.index_B.min():.1f} – {index_b.index_B.max():.1f}')
print(f'Latest value: {index_b.index_B.iloc[-1]:.1f} (base = {BASE_INDEX})')
print(f'Residual norm: {result_b[3]:.4f}')

Adjacent pairs: 575,109
Design matrix : 575,109 × 27 (sparse)


Solving OLS via lsqr...
Base period (Method B): 2025-11-17

Index range: 99.7 – 120.0
Latest value: 117.2 (base = 100.0)
Residual norm: 61.9642


In [7]:
# Overlay A and B
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(weekly_a['price_week'], weekly_a['index_A'],
        color='#546e7a', lw=1.4, ls='--', alpha=0.7, label='Method A — Chained Mean')
ax.plot(index_b['price_week'], index_b['index_B'],
        color='#1565c0', lw=2.2, label='Method B — BMN OLS')
ax.axhline(BASE_INDEX, color='#aaa', lw=0.8, ls='--')

ax.set_ylabel('Index (base = 100)', fontsize=11)
ax.set_title('MTG Repeat-Sales Price Index — Methods A vs B', fontsize=13)
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))

plt.tight_layout()
plt.savefig(DATA_DIR / 'rsi_method_AB.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Method C — All-Pairs WLS (Case-Shiller Variance Correction)

**Intuition:** The variance of a log price change grows with the time gap between observations. A card priced today and one year ago is noisier than one priced today and last week. Weight each pair inversely by the squared time gap.

**Weight function:**
$$w(i, t_1, t_2) = \frac{1}{(t_2 - t_1)^2}$$

**Uses all pairs**, not just adjacent — recovering information from cards observed infrequently.

**WLS:** β̂ = lstsq(diag(√w) X, √w ⊙ y)

**Memory guard:** Cards with > 52 weekly observations use adjacent pairs + quarterly samples (gaps of 13, 26, 39, 52 weeks) to prevent pair explosion.

In [8]:
# Method C: All-Pairs WLS (Case-Shiller)

QUARTERLY_GAPS = {13, 26, 39, 52}  # allowed long-gap sizes for dense cards

all_pairs_c = []  # list of (t1_k, t2_k, delta_log, weight)
dense_card_count = 0

for card_id, grp in df.dropna(subset=['log_price', 'week_k']).groupby('card_version_id'):
    obs = grp.sort_values('week_k')[['week_k', 'log_price']].values
    n_obs = len(obs)

    if n_obs < 2:
        continue

    is_dense = n_obs > MAX_OBS_C
    if is_dense:
        dense_card_count += 1

    for (k1, lp1), (k2, lp2) in combinations(obs, 2):
        gap = int(k2 - k1)  # gap in weeks

        # Memory guard: dense cards only keep adjacent + quarterly gaps
        if is_dense and gap != 1 and gap not in QUARTERLY_GAPS:
            continue

        delta = lp2 - lp1
        if abs(delta) > MAX_LOG_JUMP * gap:  # scaled jump guard
            continue

        all_pairs_c.append((int(k1), int(k2), delta, 1.0 / (gap * gap)))

pairs_c = pd.DataFrame(all_pairs_c, columns=['k1', 'k2', 'delta_log', 'weight'])
n_pairs_c = len(pairs_c)

print(f'Dense cards (>{MAX_OBS_C} weeks), subsampled: {dense_card_count:,}')
print(f'Total pairs for Method C: {n_pairs_c:,}')
print(f'Gap distribution:')
gap_counts = (pairs_c['k2'] - pairs_c['k1']).value_counts().sort_index()
for gap, cnt in gap_counts.head(10).items():
    print(f'  gap={gap:3d} weeks: {cnt:,} pairs')

Dense cards (>52 weeks), subsampled: 0
Total pairs for Method C: 7,587,360
Gap distribution:
  gap=  1 weeks: 572,416 pairs
  gap=  2 weeks: 546,213 pairs
  gap=  3 weeks: 521,162 pairs
  gap=  4 weeks: 496,771 pairs
  gap=  5 weeks: 472,854 pairs
  gap=  6 weeks: 449,223 pairs
  gap=  7 weeks: 425,937 pairs
  gap=  8 weeks: 403,012 pairs
  gap=  9 weeks: 380,231 pairs
  gap= 10 weeks: 357,594 pairs


In [9]:
# Build sparse design matrix for Method C
print('Building sparse design matrix...')

X_c = lil_matrix((n_pairs_c, n_cols), dtype=np.float64)

for row_idx, row in pairs_c.iterrows():
    k1, k2 = int(row['k1']), int(row['k2'])
    # +1 at column k2-1 (β_{k2}), -1 at column k1-1 (β_{k1}), skipping β₀
    if k2 - 1 >= 0:
        X_c[row_idx, k2 - 1] = +1.0
    if k1 - 1 >= 0:
        X_c[row_idx, k1 - 1] = -1.0

X_c_csr = X_c.tocsr()
print('Design matrix built.')

# Apply WLS: pre-multiply rows by sqrt(weight)
sqrt_w = np.sqrt(pairs_c['weight'].values)
W_sqrt = diags(sqrt_w)
X_c_w  = W_sqrt @ X_c_csr
y_c_w  = sqrt_w * pairs_c['delta_log'].values

print('Solving WLS via lsqr...')
result_c = lsqr(X_c_w, y_c_w, show=False)
beta_c   = result_c[0]

log_index_c = np.concatenate([[0.0], beta_c])

# Anchor to same base week as Method B
log_index_c_anchored = log_index_c - log_index_c[base_k_b]
index_c_values       = BASE_INDEX * np.exp(log_index_c_anchored)

index_c = pd.DataFrame({
    'price_week': all_weeks,
    'week_k':     range(n_weeks),
    'index_C':    index_c_values,
})

print(f'\nIndex range: {index_c.index_C.min():.1f} – {index_c.index_C.max():.1f}')
print(f'Latest value: {index_c.index_C.iloc[-1]:.1f} (base = {BASE_INDEX})')
print(f'Residual norm: {result_c[3]:.4f}')

Building sparse design matrix...


Design matrix built.


Solving WLS via lsqr...



Index range: 99.7 – 120.8
Latest value: 117.8 (base = 100.0)
Residual norm: 102.8034


In [10]:
# All three methods overlaid
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(weekly_a['price_week'], weekly_a['index_A'],
        color='#546e7a', lw=1.4, ls=':', alpha=0.7, label='A — Chained Geometric Mean')
ax.plot(index_b['price_week'], index_b['index_B'],
        color='#1565c0', lw=2.0, ls='--', label='B — BMN Adjacent OLS')
ax.plot(index_c['price_week'], index_c['index_C'],
        color='#b71c1c', lw=2.4, label='C — Case-Shiller All-Pairs WLS')
ax.axhline(BASE_INDEX, color='#999', lw=0.8, ls='--')
ax.axvline(base_week_b, color='#999', lw=0.8, ls='--', alpha=0.6)

ax.set_ylabel('Index (base = 100)', fontsize=11)
ax.set_title('MTG Repeat-Sales Price Index — All Three Methods', fontsize=13)
ax.legend(loc='upper left', fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))
ax.text(base_week_b, ax.get_ylim()[0], f' base={base_week_b.date()}',
        fontsize=8, color='#999', va='bottom')

plt.tight_layout()
plt.savefig(DATA_DIR / 'rsi_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Diagnostics

Checking index quality: coverage per week, residual distributions, and divergence between methods.

In [11]:
# 5a. Coverage: cards per week for each method
cards_per_week = df.dropna(subset=['delta_log']).groupby('price_week')['card_version_id'].nunique()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Top: index B (primary)
axes[0].plot(index_b['price_week'], index_b['index_B'],
             color='#1565c0', lw=2, label='Method B')
axes[0].plot(index_c['price_week'], index_c['index_C'],
             color='#b71c1c', lw=1.5, ls='--', alpha=0.7, label='Method C')
axes[0].axhline(BASE_INDEX, color='#aaa', lw=0.8, ls='--')
axes[0].set_ylabel('Index', fontsize=10)
axes[0].legend(fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))

# Bottom: card coverage
axes[1].bar(cards_per_week.index, cards_per_week.values,
            width=5, color='#1565c0', alpha=0.5)
axes[1].axhline(MIN_CARDS_BASE, color='#b71c1c', lw=0.8, ls='--',
                label=f'Min threshold ({MIN_CARDS_BASE})')
axes[1].set_ylabel('Cards contributing', fontsize=10)
axes[1].set_xlabel('Week', fontsize=10)
axes[1].legend(fontsize=8)

plt.suptitle('Index Values and Weekly Card Coverage', fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / 'rsi_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

In [12]:
# 5b. Residual distributions for B and C

# Reconstruct residuals for B: predicted Δlog = β_{k} - β_{k-1}
pairs_b_res = pairs_b.copy()
pairs_b_res['predicted'] = [
    log_index_b[int(k)] - log_index_b[int(k) - 1]
    for k in pairs_b_res['week_k']
]
pairs_b_res['residual'] = pairs_b_res['delta_log'] - pairs_b_res['predicted']

# Residuals for C
pairs_c_res = pairs_c.copy()
pairs_c_res['predicted'] = [
    log_index_c[int(r['k2'])] - log_index_c[int(r['k1'])]
    for _, r in pairs_c_res.iterrows()
]
pairs_c_res['residual'] = pairs_c_res['delta_log'] - pairs_c_res['predicted']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, res, label, color in zip(
    axes,
    [pairs_b_res['residual'], pairs_c_res['residual']],
    ['Method B — BMN OLS', 'Method C — Case-Shiller WLS'],
    ['#1565c0', '#b71c1c'],
):
    res_clipped = res.clip(-2, 2)
    ax.hist(res_clipped, bins=80, color=color, alpha=0.7, edgecolor='none')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'{label}\nResidual distribution (clipped ±2)', fontsize=10)
    ax.set_xlabel('Residual (log units)')
    ax.set_ylabel('Count')
    rmse = np.sqrt((res**2).mean())
    ax.text(0.98, 0.97, f'RMSE={rmse:.4f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color=color)

plt.tight_layout()
plt.savefig(DATA_DIR / 'rsi_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [13]:
# 5c. Method divergence: A vs B vs C
compare = (
    weekly_a[['price_week', 'index_A']]
    .merge(index_b[['price_week', 'index_B']], on='price_week', how='inner')
    .merge(index_c[['price_week', 'index_C']], on='price_week', how='inner')
)
compare['A_vs_B'] = compare['index_A'] - compare['index_B']
compare['B_vs_C'] = compare['index_B'] - compare['index_C']

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(compare['price_week'], compare['A_vs_B'],
        color='#546e7a', lw=1.4, label='A − B (composition bias in A)')
ax.plot(compare['price_week'], compare['B_vs_C'],
        color='#e65100', lw=1.4, label='B − C (variance correction in C)')
ax.axhline(0, color='#aaa', lw=0.8, ls='--')
ax.set_ylabel('Index point difference', fontsize=10)
ax.set_title('Divergence Between Methods', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(DATA_DIR / 'rsi_divergence.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSummary statistics of divergence:')
print(compare[['A_vs_B', 'B_vs_C']].describe().to_string())


Summary statistics of divergence:
             A_vs_B     B_vs_C
count  2.700000e+01  27.000000
mean  -2.011994e-11   0.212056
std    3.317991e-08   0.529731
min   -7.093996e-08  -0.845407
25%   -2.286960e-09   0.014900
50%    0.000000e+00   0.266983
75%    4.145669e-09   0.658340
max    7.090090e-08   0.815505


## 6. Sub-Indices by Rarity (Method B)

Run Method B independently for each rarity tier. Commons/uncommons tend to be bulk assets; rares and mythics are more volatile and more investment-relevant.

This segmentation is the first step toward a **rarity-weighted composite index** in V2.

In [14]:
RARITY_ORDER  = ['mythic', 'rare', 'uncommon', 'common']
RARITY_COLORS = {'mythic': '#e07b39', 'rare': '#c5a800',
                 'uncommon': '#607d8b', 'common': '#78909c'}

def build_bmn_index(pairs_df, all_weeks_list, base_k):
    """Run BMN OLS on a filtered pairs DataFrame; return index values array."""
    n_w = len(all_weeks_list)
    n_c = n_w - 1
    n_p = len(pairs_df)
    if n_p < n_c:
        return None

    X = lil_matrix((n_p, n_c), dtype=np.float64)
    for i, (_, row) in enumerate(pairs_df.iterrows()):
        k = int(row['week_k'])
        if k - 1 >= 0: X[i, k - 1] = +1.0
        if k - 2 >= 0: X[i, k - 2] = -1.0

    beta, *_ = lsqr(X.tocsr(), pairs_df['delta_log'].values)
    log_idx  = np.concatenate([[0.0], beta])
    log_idx -= log_idx[base_k]
    return BASE_INDEX * np.exp(log_idx)


rarity_indices = {}
for rarity in RARITY_ORDER:
    r_cards = df[df['rarity'] == rarity]['card_version_id'].unique()
    r_pairs = pairs_b[pairs_b['card_version_id'].isin(r_cards)].copy()
    if len(r_pairs) < 100:
        print(f'{rarity}: too few pairs ({len(r_pairs)}) — skipped')
        continue
    idx = build_bmn_index(r_pairs, all_weeks, base_k_b)
    if idx is not None:
        rarity_indices[rarity] = idx
        print(f'{rarity:10s}: {len(r_pairs):,} pairs | latest={idx[-1]:.1f}')

# 4-panel plot
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=False)
axes = axes.flatten()

for ax, (rarity, idx_vals) in zip(axes, rarity_indices.items()):
    color = RARITY_COLORS.get(rarity, '#888')
    ax.plot(all_weeks, idx_vals, color=color, lw=2)
    ax.axhline(BASE_INDEX, color='#aaa', lw=0.7, ls='--')
    ax.set_title(f'{rarity.capitalize()} (Method B)', fontsize=11, color=color)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))
    ax.set_ylabel('Index (base=100)', fontsize=9)

for ax in axes[len(rarity_indices):]:
    ax.set_visible(False)

plt.suptitle('MTG Repeat-Sales Index by Rarity — Method B (BMN OLS)', fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / 'rsi_by_rarity.png', dpi=150, bbox_inches='tight')
plt.show()

mythic    : 110,059 pairs | latest=113.5


rare      : 314,912 pairs | latest=117.0


uncommon  : 81,772 pairs | latest=122.0


common    : 65,477 pairs | latest=118.3


## 7. Summary Table & Export

In [15]:
# Assemble final output table
final = (
    weekly_a[['price_week', 'index_A', 'n_cards']]
    .merge(index_b[['price_week', 'index_B']], on='price_week', how='outer')
    .merge(index_c[['price_week', 'index_C']], on='price_week', how='outer')
    .sort_values('price_week')
    .reset_index(drop=True)
)

# Add rarity sub-indices
for rarity, idx_vals in rarity_indices.items():
    rarity_df = pd.DataFrame({'price_week': all_weeks, f'index_B_{rarity}': idx_vals})
    final = final.merge(rarity_df, on='price_week', how='left')

final.to_parquet(DATA_DIR / 'rsi_final_index.parquet')
print(f'Saved {len(final)} weeks to data/rsi_final_index.parquet')
print()

# Summary stats
print('=== INDEX SUMMARY ===')
print(f'Period        : {final.price_week.min().date()} → {final.price_week.max().date()}')
print(f'Weeks         : {len(final)}')
print(f'Base period   : {base_week_b.date()} = 100')
print()

for col, label in [('index_A','Method A'), ('index_B','Method B'), ('index_C','Method C')]:
    s = final[col].dropna()
    print(f'{label:12s} | latest={s.iloc[-1]:.1f} | min={s.min():.1f} | max={s.max():.1f} | '
          f'total return={(s.iloc[-1]/BASE_INDEX - 1)*100:+.1f}%')

Saved 28 weeks to data/rsi_final_index.parquet

=== INDEX SUMMARY ===
Period        : 2025-11-10 → 2026-05-18
Weeks         : 28
Base period   : 2025-11-17 = 100

Method A     | latest=117.2 | min=100.0 | max=120.0 | total return=+17.2%
Method B     | latest=117.2 | min=99.7 | max=120.0 | total return=+17.2%
Method C     | latest=117.8 | min=99.7 | max=120.8 | total return=+17.8%


## 8. Interpretation & Next Steps

### What this index measures
The index tracks **listed market price appreciation** for a basket of NM nonfoil MTG cards ≥ $1 on TCGPlayer. It is a **price index, not a total-return index** — it does not include transaction costs, storage, or the emotional yield of playing with the cards.

### Reading the divergence between methods
- **A vs B:** If A > B, Method A's equal-weighting is overrepresenting cheap, noisy cards whose prices moved up more than the typical card. If A < B, the reverse. Large persistent divergence = composition bias is material.
- **B vs C:** If B > C, short-gap adjacent pairs were "lucky" — cards briefly spiked and settled. The long-gap pairs in C anchor the index to a more stable long-run level. Divergence typically grows in thin market periods (post-rotation, low tournament activity).

### Benchmarking (TODO)
Compare the total return from `base_week` to `latest_week` against:
- S&P 500: download `^GSPC` from Yahoo Finance and compute the return over the same period
- CPI: adjust for inflation to get **real** MTG returns
- Pokémon / sports cards: PriceCharting public data

### Upgrade path
| Enhancement | Status |
|---|---|
| Foil sub-index | Pending — data is sparse, needs dedicated cleaning |
| Confidence intervals via bootstrap | Pending — resample pairs B ×200, report 95% CI |
| Format sub-indices (Standard / Commander / Legacy) | Pending — join format legality from Scryfall |
| Multi-source index (TCGPlayer + Cardmarket) | Pending — currency harmonisation layer needed |
| Wishlist lead-time correlation | Pending — wishlist snapshot accumulation required |
| API endpoint for the index | Pending — expose via `/api/v1/analytics/price-index` |

**Reference:** `docs/domain/MTG_QUANT_RESEARCH.md` § 6 (Open Research Gaps)